In [102]:
# Imports
%pip install prettytable
from ipywidgets.widgets import FloatSlider
from ipywidgets import interact
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Union
from pathlib import Path
from prettytable import PrettyTable
from improutils import *


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [103]:
# Kalibrace knihovna
def camera_calibration(calib_path: str, 
                       chess_shape: Tuple[int,int], 
                       cv2_flags:int = 0, 
                       extensions: List[str] = ["jpg", "jpeg" ,"png", "tiff", "bmp"]) -> Tuple[float, 
                                                                                        np.ndarray,
                                                                                        np.ndarray, 
                                                                                        Tuple[np.ndarray], 
                                                                                        Tuple[np.ndarray], 
                                                                                        np.ndarray, 
                                                                                        np.ndarray, 
                                                                                        np.ndarray, 
                                                                                        Dict[str,np.ndarray]]: 
    """Calibrates camera from images with chessboard pattern, using OpenCV's cv2.calibrateCameraExtended function

    Args:
        calib_path (str): path to the folder containing chessboard pattern images
        chess_shape (Tuple[int,int]): interior corner count in the format of rows, columns
        cv2_flags (int, optional): additional OpenCV's flags for cv2.calibrateCameraExtended. Defaults to 0.
        extensions (List[str], optional): allowed image extensions. Defaults to ["jpg", "jpeg" ,"png", "tiff"].

    Raises:
        ValueError: if calibration images have different sizes
        ValueError: if no calibration images were found or could not be read from the provided path 
        ValueError: if no chessboard patterns were detected in the images
        
    Returns:
        Tuple[float, np.ndarray, np.ndarray, Tuple[np.ndarray], Tuple[np.ndarray], np.ndarray, np.ndarray, np.ndarray, Dict[str,np.ndarray]]: 
        returns the output from cv2.calibrateCameraExtended and dictionary with image names as keys and images with drawn chessboard corners as values
    """
    print(f"Processing images from {calib_path} with possible extensions {extensions}")
    def correct_extension(path, extensions):
        return path.is_file() and path.suffix[1:].lower() in extensions
    # termination criteria for subpixel corner detection
    # by default it is set to 30 iterations and epsilon = 0.001
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    
    # prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
    objp = np.zeros((chess_shape[0] * chess_shape[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:chess_shape[0], 0:chess_shape[1]].T.reshape(-1, 2)

    # arrays to store object points and image points from all the images.
    objpoints = [] # 3D point in real world space
    imgpoints = [] # 2D points in image plane.

    image_paths = sorted([path for path in Path(calib_path).glob("*") if correct_extension(path,extensions)]) 
    chess_brd_images = 0
    read_images = 0
    chessboard_images = {}
    img_size = None
    for img_path in image_paths:
        img_name = img_path.namevyberete si z kamer a objektivů na vašem stole, co se vám hodí. Do notebooku musíte napsat co jste vybrali a proč.
        
        img = cv2.imread(img_path)
        if img is None:
            print(f"File {img_name} could not be read, skipping...")
            continue
        else:
            read_images += 1
            if img_size is None:
                # need to be in the format of width, height
                img_size = img.shape[:2][::-1]
            else:
                if img_size != img.shape[:2][::-1]:
                    raise ValueError("All images must have same size.")
            print(f"File {img_name} is being processed...")

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # find the chess board corners
        ret, corners = cv2.findChessboardCorners(gray, chess_shape, None)

        # if found, add object points, image points (after refining them)
        if ret:
            chess_brd_images += 1
            print(f"\t Corners found!")
            objpoints.append(objp)
            subpix_corners = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
            imgpoints.append(subpix_corners)

            chessboard_images[img_name] = cv2.drawChessboardCorners(img, chess_shape, subpix_corners, ret)

        else:
            print(f"\t Corners NOT found!")
            continue
    
    print(f"Number of images with detected chessboard: {chess_brd_images}/{read_images}")

    if read_images == 0:
        raise ValueError("No images were read from the provided path.")
    
    if chess_brd_images == 0:
        raise ValueError("No chessboard patterns were detected in the images.")

    calib_values = cv2.calibrateCameraExtended(objpoints, imgpoints, img_size, cameraMatrix=None, distCoeffs=None, flags=cv2_flags, criteria=criteria)
    reprojection_error, camera_matrix, dist_coeffs, rvecs, tvecs, std_deviations_intrinsics, std_deviations_extrinsics, per_view_errors = calib_values
    return reprojection_error, camera_matrix, dist_coeffs, rvecs, tvecs, std_deviations_intrinsics, std_deviations_extrinsics, per_view_errors, chessboard_images
    
def calibration_stats(reprojection_error:float,
                      camera_matrix: np.ndarray, 
                      dist_coeffs:np.ndarray, 
                      std_deviations_intrinsics:np.ndarray=None,
                      per_view_errors:np.ndarray=None, 
                      view_names:List[str]=None, 
                      pixel_size:Union[float,Tuple[float,float]]=None) -> None:
    """Prints calibration statistics using. 
    RMS re-projection error, estimated intrinsics and distortion parameters, standard deviations of intrinsics, focal length in millimeters and per view reprojection errors.

    Args:
        reprojection_error (float): re-projection error from cv2.calibrateCamera
        camera_matrix (np.ndarray): camera matrix from cv2.calibrate
        dist_coeffs (np.ndarray): distortion coefficients from cv2.calibrateCamera
        std_deviations_intrinsics (np.ndarray, optional): std_deviations_intrinsics from cv2.calibrateCameraExtended. Defaults to None.
        per_view_errors (np.ndarray, optional): per_view_errors from cv2.calibrateCameraExtended. Defaults to None.
        view_names (List[str], optional): image names for which we detected the chessboard. Defaults to None.
        pixel_size (Union[float,Tuple[float,float]], optional): size of physical pixels of a camera in micrometers eg. 4.8 or 5.86 or [5.86, 4.8] for non square pixels. Defaults to None.
    """
    # opencv always returns atleast 4 distortion coefficients
    params_amount = 4 + dist_coeffs.shape[1]
    
    parameters = ["fx", "fy", "cx", "cy", "k1", "k2", "p1", "p2", "k3", "k4", "k5", "k6", "s1", "s2", "s3", "s4", "Tx", "Ty"]
    units = ["pixels"] * 4 + ["unitless"] * (params_amount - 4)
    
    print(f"RMS re-projection error: {reprojection_error:.5f} pixels")
    
    print(f"\nEstimated intrinsics parameters")
    intrinsics_table = PrettyTable()
    intrinsics_table.add_column("Parameter", parameters[:4])
    intrinsics_table.add_column("Estimated Value", [f"{val:.5f}" for val in [camera_matrix[0, 0], camera_matrix[1, 1], camera_matrix[0, 2], camera_matrix[1, 2]]])
    intrinsics_table.add_column("Unit", units[:4])
    print(intrinsics_table)
    
    print(f"\nEstimated Distortion parameters")
    distortion_table = PrettyTable()
    distortion_table.add_column("Parameter", parameters[4:params_amount])
    distortion_table.add_column("Distortion", [f"{val:.5f}" for val in dist_coeffs[0, :params_amount-4]])
    distortion_table.add_column("Unit", units[4:params_amount])
    print(distortion_table)
    
    if std_deviations_intrinsics is not None:
        print(f"\nIntrinsic parameters standard deviation")
        intrinsics_std_table = PrettyTable()
        intrinsics_std_table.add_column("Parameter", parameters[:params_amount])
        intrinsics_std_table.add_column("Value", [f"±{val:.5f}" for val in std_deviations_intrinsics[:params_amount,0]])
        intrinsics_std_table.add_column("Unit", units[:params_amount])
        print(intrinsics_std_table)
        
    if pixel_size is not None and std_deviations_intrinsics is not None:
        if not isinstance(pixel_size, tuple):
            pixel_size = (pixel_size, pixel_size)
        print(f"\nEstimated Focal length in millimeters")
        focal_length_table = PrettyTable()
        focal_length_table.add_column("Parameter", parameters[:2])
        focal_length_table.add_column("Value ± Std Deviation", [f"{val*pix_size/1000:.5f} ± {std*pix_size/1000:.5f}" for val, pix_size, std in zip([camera_matrix[0, 0], camera_matrix[1, 1]], pixel_size, std_deviations_intrinsics[:2,0])])
        focal_length_table.add_column("Unit", ["millimeter"] * 2)
        print(focal_length_table)

    if per_view_errors is not None:
        print(f"\nPer view reprojection errors")
        view_error_table = PrettyTable()
        # Sort the view names and errors by the errors in descending order
        sorted_views_and_errors = sorted(zip(view_names, per_view_errors[:,0]), key=lambda x: x[1], reverse=True)
        sorted_view_names, sorted_errors = zip(*sorted_views_and_errors)
        view_error_table.add_column("Image name", sorted_view_names)
        view_error_table.add_column("Re-projection error (sorted)", [f"{val:.5f}" for val in sorted_errors])
        view_error_table.add_column("Unit", ["pixels"] * len(sorted_view_names))
        print(view_error_table)

def correct_frame(img, camera_matrix, dist_coeffs):
    """Returns undistorted image."""
    return cv2.undistort(img, camera_matrix, dist_coeffs)

def _plot_grid(xv, yv, squares, ax):
    for i  in np.linspace(0, xv.shape[1] - 1, squares+1, dtype=int):
        ax.plot(xv[i,:], yv[i,:], 'k-')
    for j in np.linspace(0, xv.shape[0] - 1, squares+1, dtype=int):
        ax.plot(xv[:,j], yv[:,j], 'k-')
    
    ax.axis('off')

def _radial_distortion(xv, yv, k):
    xv_radial = np.zeros_like(xv)
    yv_radial = np.zeros_like(yv)
    for i in range(xv.shape[0]):
        for j in range(xv.shape[1]):
            r = np.sqrt(xv[i,j]**2 + yv[i,j]**2)
            radial = (1 + (k[0]*(r**2) + k[1]*(r**4) + k[2]*(r**6)))/(1 + (k[3]*(r**2) + k[4]*(r**4) + k[5]*(r**6)))
            xv_radial[i,j] = xv[i,j]*radial
            yv_radial[i,j] = yv[i,j]*radial
    return xv_radial, yv_radial

def _tangetial_distortion(xv, yv, p):
    xv_tang = np.zeros_like(xv)
    yv_tang = np.zeros_like(yv)
    for i in range(xv.shape[0]):
        for j in range(xv.shape[1]):
            x = xv[i,j]
            y = yv[i,j]
            r = np.sqrt(x**2 + y**2)
            x_tang = x + (2*p[0]*x*y + p[1]*(r**2 + 2*x**2))
            y_tang = y + (p[0]*(r**2 + 2*y**2) + 2*p[1]*x*y)
            xv_tang[i,j] = x_tang
            yv_tang[i,j] = y_tang
    return xv_tang, yv_tang

def plot_distortion(k1:float,k2:float,k3:float,k4:float,k5:float,k6:float, p1:float,p2:float) -> None:
    """Plots radial, tangential and compounded (radial + tangential) distortion grid. Using the Brown-Conrady model.

    Args:
        k1 (float): radial distortion coefficient
        k2 (float): radial distortion coefficient
        k3 (float): radial distortion coefficient
        k4 (float): radial distortion coefficient
        k5 (float): radial distortion coefficient
        k6 (float): radial distortion coefficient
        p1 (float): tangential distortion coefficient
        p2 (float): tangential distortion coefficient
    """
    k = (k1,k2,k3,k4,k5,k6)
    p = (p1,p2)
    squares = 10 # amount of squares in the grid
    pts = 100
    # realistical values for image with 2500 x 2500 pixels with focal length of 35mm which is close to 10500 pixels with basler camera pixel size, origin is in the center - therfore x and y should be within values +-(2500/10500)/2
    width = 0.23
    height = 0.23
    xv, yv = np.meshgrid(np.linspace(-width/2,width/2,pts), np.linspace(-height/2,height/2,pts))

    xv_radial, yv_radial = _radial_distortion(xv, yv, k)
    xv_tang, yv_tang = _tangetial_distortion(xv, yv, p)

    _, axs = plt.subplots(1, 3, figsize=(15, 5))

    _plot_grid(xv_radial, yv_radial, squares, axs[0])
    axs[0].set_title('Radial distortion grid')

    _plot_grid(xv_tang, yv_tang, squares, axs[1])
    axs[1].set_title('Tangential distortion grid')

    _plot_grid(xv_radial + xv_tang, yv_radial + yv_tang, squares, axs[2])
    axs[2].set_title('Compounded distortion grid')
    plt.show()

# 1 ) Kamera, objektiv, osvětlení

## Kamera
Výpočet potřebného rozlišení snímače:

Zadaná přesnost: 0.3mm

Odhad velikosti předmětu: 80mm

Potřebné rozlišení získáme jako $(80*1.1/0.3)*2 = 600px$

Potřebujeme tedy kameru s rozlišením alespoň 600px na delší straně.

Pokud není nezbytně nutné použít bayerovu masku která zajištuje barevný obraz, použiju monochromatickou kameru, která má lepší vlastnosti co se přesnosti týče

## Objektiv
Vybral jsem ac4200-180gm. - maticova ,rozliseni, framerate, GiGE, monochrom

Nyní je třeba vybrat objektiv. Vycházíme ze vzorečku $Y/L = Y'/f$.

Hledám objektiv, aby jeho údaje minimalizovaly overlap. Minimalní vzdálenost a ohnisková vzdálenost jsou parametry objektivu.
 ideální vybrat 12mm se vzfdáleností 100

 velikost snímače se nebere diagonální

Zpětným dosazením mi vyšlo, že ideální pracovní vzdálenost je 80mm. 

## Osvětlění 
Přímé světlo bez difúzoru pro matné objekty.

S difúzorem na lesklé objekty.

Pocitani lentilek je dobry dom - minimalizuje odlesky

Boční světlo pro čtení rytin.

BACKLIGHT pro získání obrysu předmětu.

## A - B
Červená - Zelená

Modrá - Oranžová

Fialová - Žlutá

Když na objekt barvy A zasvítím světlem A a budu snímat pomocí monochromatické (černobílé) kamery, bude světlý. Pokud na něj zasvítím světlem B, bude se jevit tmavým. A naopak. 

# 2) Parametry snímání

clona - Clonu jsem uzavřel tak, abych co nejvíce zvýšil hloubku ostrosti, ale ještě nebyla znatelná vada difrakce a na snímač dopadalo dostatečné množství světla.

gain - S gainem se snažím pracovat pouze minimálně, abych nepřidal do výsledného obrázku zbytečný šum.

expoziční čas - Nastaveno na hodnotu XXX, hodnotu jsem šteloval v závislosti na cloně, abych do snímače dostal dostatek světla.

vyvážení bílé - Bílou barvu jsem vyvážil jednorázově pomocí čistého papíru.

ostrost - Objektiv jsem zaostřil podle odhadové ostřící vzdálenost na objektivu. Korekci jsem udělal pomocí pylonvieweru.

vady objektivů - Žádná zaznamenaná vada. 
může být difrakce - řeší se kalibrací
vinětace
distorze
chromatická aberace - nevyskytuje se u monochromatických kamer



přidat screenshot parametrů


# 3) Kalibrace kamery
Nasnímal jsem xx obrázků a kalibraci jsem provedl pomocí knihovních funkcí. Na nasnímaný obrázek aplikuju inverzní distorzní funkci, kterou jsem dostal ze samotné kalibrace.

Důležitý je mít ruzné snímky a různě nahnuté a zaostřené - k tomu slouží poslední výpis. - tam ty hodnoty na pixel musi byt cca stejny

během kalibrace můžu hýbat jenom s clonou

In [104]:
# Kalibrace kamery
# calib_folder_path = "./calib"

# chess_shape = (14,9) 
# reprojection_error, camera_matrix, dist_coeffs, _, _, std_deviations_intrinsics, _, per_view_errors, chessboard_images = camera_calibration(calib_folder_path, chess_shape) ###

In [105]:
# Zobrazení
# detected_images = list(chessboard_images.values())
# plot_images(*detected_images)

In [106]:
# Statistiky
# detected_image_names = list(chessboard_images.keys())
# pixel_size = 5.86 ### use cameras datasheet to find physical pixel size of the sensor
# calibration_stats(reprojection_error, camera_matrix, dist_coeffs, std_deviations_intrinsics, per_view_errors, detected_image_names, pixel_size)

In [107]:
# Aplikace inverzní distorzní funkce na snímek
# img = load_image("./KEY.bmp")
# img_corrected = correct_frame(img, camera_matrix, dist_coeffs) ###
# plot_images(img, titles=["Distorted image"]) ### 
# plot_images(img_corrected, titles=["Undistorted image"]) ###

# 4) Segmentace a měření součástky (p_min = 0,1mm) [6b]

zachovat stejně soustavu a udělat dva separátní snímky kostky a objektu.

nejdřív použít kalibraci na obrázek

chyba může vzniknout když si drbnu do sestavy nebo nedobrým zaostřením nebo i referenční kostka může být špatná


In [108]:


# img = load_image("segmentace.bmp")

# # print size of the image
# print(f"Image size: {img.shape[1]}x{img.shape[0]} pixels")

# hsv_img = to_hsv(img)
# # print size of the hsv image
# print(f"HSV image size: {hsv_img.shape[1]}x{hsv_img.shape[0]} pixels")

In [109]:
# img = load_image("segmentace.bmp")

# mask = None

# def create_slider(min, max, description):
#     description = description.ljust(30, '\xa0')
#     return widgets.IntRangeSlider( min=min, max=max, step=1,value=[min,max], 
#                                    description=description, 
#                                    continuous_update=False, 
#                                    orientation='horizontal',
#                                    style=dict(description_width='initial'),
#                                    layout=widgets.Layout(width='auto'),
#                                   )

# @interact(red=create_slider(min=0, max=255, description='Hue:'),
#           green=create_slider(min=0, max=255, description='Saturation:'),
#           blue=create_slider(min=0, max=255, description='Value:'))
# def _(red, green, blue):

#     lower_bound = (red[0], green[0], blue[0])
#     upper_bound = (red[1], green[1], blue[1])

#     mask = segmentation_two_thresholds(hsv_img,lower_bound,upper_bound) ###

#     applied_img = apply_mask(hsv_img, mask)

#     plot_images(mask, applied_img, titles=["Mask", "Applied mask"])

In [110]:
# lower_bound = (0, 92, 26)
# upper_bound = (87, 255, 105)

# mask = segmentation_two_thresholds(hsv_img,lower_bound,upper_bound)

# plot_images(mask, titles=["Mask"])

# # print size of the mask
# print(f"Mask size: {mask.shape[1]}x{mask.shape[0]} pixels")

In [111]:
# img_contours, count, contours = find_contours(mask, min_area=100)
# plot_images(img_contours, titles=["Contours"])

In [112]:
# width_real = 80 ###
# height_real = 40 ###

# # print(contours)
# # find width and height of the contour
# width_pic = None
# height_pic = None

# if len(contours) > 1:
#     raise ValueError("BACHA CO DELAS.")

# for c in contours:
#     box = cv2.minAreaRect(c)
#     print(f"Contour: Width: {box[1][0]}, Height: {box[1][1]}")
#     width_pic = box[1][0]
#     height_pic = box[1][1]

# # print area of the contour
# area = cv2.contourArea(contours[0])
# print(f"Area of the contour: {area} pixels")

In [113]:
# ratio = width_real / width_pic
# print(f"Ratio: {ratio:.2f} (real width / picture width)")
# # KONTROLA 2. PARAMETRU
# ratio_2 = height_real / height_pic
# print(f"Ratio: {ratio_2:.2f} (real height / picture height)")

In [114]:
# img = load_image("segmentace.bmp")

# mask = None

# def create_slider(min, max, description):
#     description = description.ljust(30, '\xa0')
#     return widgets.IntRangeSlider( min=min, max=max, step=1,value=[min,max], 
#                                    description=description, 
#                                    continuous_update=False, 
#                                    orientation='horizontal',
#                                    style=dict(description_width='initial'),
#                                    layout=widgets.Layout(width='auto'),
#                                   )

# @interact(red=create_slider(min=0, max=255, description='Red:'),
#           green=create_slider(min=0, max=255, description='Green:'),
#           blue=create_slider(min=0, max=255, description='Blue:'))
# def _(red, green, blue):

#     lower_bound = (red[0], green[0], blue[0])
#     upper_bound = (red[1], green[1], blue[1])

#     mask = segmentation_two_thresholds(img,lower_bound,upper_bound) ###

#     applied_img = apply_mask(img, mask)

#     plot_images(mask, applied_img, titles=["Mask", "Applied mask"])

In [115]:
# lower_bound = (90, 42, 98)
# upper_bound = (255, 132, 202)

# mask = segmentation_two_thresholds(img,lower_bound,upper_bound)

# plot_images(mask, titles=["Mask"])


In [116]:
# img_contours, count, contours = find_contours(mask, min_area=100)
# plot_images(img_contours, titles=["Contours"])

In [117]:
# # print(contours)
# # find width and height of the contour
# width_pic = None
# height_pic = None

# for c in contours:
#     box = cv2.minAreaRect(c)
#     print(f"Contour: Width: {box[1][0]}, Height: {box[1][1]}")
#     width_pic = box[1][0]
#     height_pic = box[1][1]
#     real_width = width_pic * ratio
#     real_height = height_pic * ratio_2
#     print(f"Real width: {real_width:.2f} mm, Real height: {real_height:.2f} mm")

vykresleni

In [ ]:
## Vykresleni
# cv2.line(image, start_point, end_point, color, thickness) 
# contour_drawn = cv2.drawContours(img_patterns.copy(), contours, -1, color=(255, 0, 0 ), thickness=10)

# 5) Losovaná úloha:

existuje cropovani

otoceni obrazku

In [1]:
# # Získání středu obrázku
# (h, w) = img.shape[:2]
# center = (w // 2, h // 2)

# # Vytvoření rotační matice
# angle = 27
# scale = 1.0  # Můžeš zvětšit/zmenšit obrázek
# M = cv2.getRotationMatrix2D(center, angle, scale)

# # Aplikuj otočení
# rotated = cv2.warpAffine(img, M, (w, h))  ###borderValue=(r,g,b)

spocitej uhel

In [ ]:
# def get_angle(p1, p2, p3):
#     v1 = np.subtract(p2, p1)
#     v2 = np.subtract(p2, p3)
#     cos = np.inner(v1, v2) / la.norm(v1) / la.norm(v2)
#     rad = np.arccos(np.clip(cos, -1.0, 1.0))
#     return np.rad2deg(rad)

nalezeni skrabance - namapovat objekt na referencni objekt

In [ ]:
## posun obrazku na stejne misto
# orb = cv2.ORB_create(5000)
# kp1, des1 = orb.detectAndCompute(gray1, None)
# kp2, des2 = orb.detectAndCompute(gray2, None)

# bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
# matches = bf.match(des1, des2)
# matches = sorted(matches, key=lambda x: x.distance)

# src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
# dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts)

# aligned = cv2.warpAffine(image2, M, (image1.shape[1], image1.shape[0]))

## rozdil obrazku
# diff = cv2.absdiff(img_clean, img_scratch)

# CVIKO 6 - Použití transformační matice

In [118]:
# import os
# import io
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# from improutils import *
# %matplotlib inline
# np.set_printoptions(formatter={'float': lambda x: "{0:0.3f}".format(x)})

# img = load_image('images/ctverce.bmp')
# plot_images(img)

# # Vybrání referenčních bodů pro zjištění perspektivy a získání transformační matice
# src_pts = np.array([(825,254), (1651,586),(410,665),(1342,1152)])
# dst_pts = np.array([(0,0),(2970,0), (0, 2100), (2970,2100)])
# H, mask = cv2.findHomography(src_pts, dst_pts)

# # Narovnání obrazu
# tmp = img
# hh, ww = img.shape[:2]
# warped_img = cv2.warpPerspective(img, H, (2970, 2100))
# plot_images(warped_img)

# # Měření přímky
# line_start = np.array([[(718, 579)]], dtype='float32')
# line_end = np.array([[(1267, 703)]], dtype='float32')
# line_start_t = cv2.perspectiveTransform(line_start, H)
# line_end_t = cv2.perspectiveTransform(line_end, H)
# print(line_start_t)
# dist = math.sqrt((line_start_t[0][0][0]-line_end_t[0][0][0])**2+(line_start_t[0][0][1]-line_end_t[0][0][1])**2)
# print(f'D = {dist / 10:.02f} mm')

# 9 Fourierova transformace

In [119]:
## načtení a crop
# cardboard_texture = load_image('data/cardboard.bmp') ### načtení dat
# cardboard_texture = to_gray(cardboard_texture) ### převod do šedotónu
# cardboard_texture_cropped = crop(cardboard_texture, 200, 1000, 700, 1500) # je nutné oříznout na čtverec

# # Filtrace medianem
# cardboard_texture_filtered = filtration_median(cardboard_texture_cropped, 41)
# cardboard_normalized = normalize(cardboard_texture_filtered) ###
# plot_images(cardboard_normalized)

# # Aplikace Fourierovy transformace
# mag_spec, spectrum_shift = apply_fft(cardboard_normalized) ### aplikujte Fourierovu transformaci
# plot_images(cardboard_normalized, mag_spec)

# # Nalezení největších vrcholů ve spektru
# # Segmentace
# spectrum_intresting = segmentation_one_threshold(mag_spec, 215)
# # Nalezení indexů
# idx = cv2.findNonZero(spectrum_intresting)
# plot_images(cardboard_normalized, spectrum_intresting)

# # Tvoření filtru podle hodnot
# filter_mask = create_filter_mask(cardboard_normalized.shape, [223, 241, 250, 259, 277], [249, 250, 251])
# spec_mask_filt = filter_mag_spec(mag_spec, filter_mask)
# cardboard_filt = inverse_fft(spectrum_shift, filter_mask)
# plot_images(cardboard_normalized, spec_mask_filt, cardboard_filt, normalize=True)

# # Hledání IDEÁLNÍCH frekvencí ve spektru
# # Podpůrné indexy a hodnoty
# indexes = idx[:, 0, 1]
# idx_middle = math.floor(len(indexes)/2)
# # Hodnota pixelu indexu stejnosměrné složky
# ind_ss = indexes[idx_middle]
# indexes = np.delete(indexes, idx_middle)
# # Maximální perioda v obraze T_max (rozlišení obrazu) [pix]
# T_max = cardboard_normalized.shape[0] ### získaná hodnota
# # Perioda opakování struktury (vzoru) [pix}
# T_vzor = 56 ### změřená hodnota
# # Frekvence vzoru [pix^-1]
# f_vzor = 1 / T_vzor ### vzorec
# # Hledaná hodnota frekvence se pozná podle vzdálenosti od středu spektra [pix]
# pix_spect = f_vzor * T_max ### vzorec

# # Hledání ideální frekvence ze zjištěných možných
# diffs = []
# for i in indexes:
#     pix_from_center = abs(i - ind_ss)
#     if pix_from_center != 0:
#         diff = abs(pix_spect - pix_from_center)
#         diffs.append(diff)

# diffs = np.array(diffs)
# ind = indexes[np.where(diffs == diffs.min())]
# print('Správné obrazové frekvence leží na hodnotách pixelů na ose y: ' + str(ind))
# print('Spektrum je totiž souměrné podle středu.')

# # Detekce defektů
# # Pomocná metoda pro vizualizaci
# def vizualize(image, image_binary):
#     _, _, contours = find_contours(image_binary, 60, fill=False)

#     result = to_3_channels(image)
#     cv2.drawContours(result, contours, -1, (0, 0, 255), 4)
#     return result

# defects_bin = segmentation_two_thresholds(cardboard_filt_final, 90, 110) ### vhodná metoda segmentace
# defects_viz = vizualize(cardboard_texture_cropped, defects_bin) 
# plot_images(defects_bin, defects_viz)

# 10 Detekce závitů

In [ ]:
# image_trans = warp_to_cartesian(image_grey) ### transformace z polárních souřadnic do kartézských
# image_trans = warp_to_polar(image_grey) ### transformace z kartézských souřadnic do polárních
# 
# img_filtered = filtration_median(img_crop, 1) ### MAGICKÁ KONSTANTA 1 NEDĚLÁ NIC, ALE VYŠŠÍ HODNOTY FUNGUJOU 
# img_edges = cv2.Canny(img_filtered, 200, 400) ### hranový detektor s vhodnými parametry
# plot_images(img_edges)

# def draw_lines(img, lines):
#     img_lines = to_3_channels(img)

#     for line in lines:
#         l = line[0]        
#         cv2.line(img_lines, (l[0], l[1]), (l[2], l[3]), (0,0,255), 2, cv2.LINE_AA)

#     return img_lines

# # Houghova transformace pro detekci přímek, nebo jiných jednoduchých tvarů
# linesP = cv2.HoughLinesP(img_edges, rho=1, theta=np.pi/180, threshold=150, minLineLength=300, maxLineGap=100)
# img_lines = draw_lines(img_edges, linesP) ### kreslení
# plot_images(img_lines)

# 12 Rozpoznávání objektů

In [121]:
# # nasnímání jak objektu tak pozadí ( na BACKLIGHTU )
# functions = [form_factor, roundness, aspect_ratio, convexity, solidity, compactness, extent]

# image = load_image('data/object_with_backlight.png')
# background = load_image('data/background.png')

# # segmentace objektu z pozadí
# def binary_mask_from_background(background_image):
#     mask = segmentation_auto_threshold(background_image)
#     return mask

# mask = binary_mask_from_background(background)
# # plot_images(background, mask)

# # aplikace masky na objekt
# def segment_object(image, background_mask):
#     image_masked = apply_mask(image, background_mask)
#     image_thresh = segmentation_auto_threshold(image_masked)
#     image_filled = fill_holes(image_thresh)
#     image_final, _, contours = find_contours(image_filled)
#     return image_final, contours[0]

# image_segmented, contour = segment_object(image, mask)
# # plot_images(image_segmented)

# # výpočet popisných statistik
# shape_descriptions = list()
# for func in functions:
#     shape_descriptions.append(func(contour))

# shape_descriptions = np.array(shape_descriptions)
# assert len(shape_descriptions) == 7, 'Nebylo použito správné množství funkcí'

# # klasifikace objektu z databáze:
# results = [np.linalg.norm(db[key] - shape_descriptions) for key in db]
# result = min(results)
# result_key = list(db.keys())[np.argmin(results)]

# print(f'Jedná se o objekt: {result_key} (vzdálenost = {result:.4f})')
# # plot_images(image_segmented)